In [6]:
# !pip install jaxns

In [1]:
%load_ext autoreload
%autoreload 2
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
from numpyro.contrib.nested_sampling import NestedSampler
from jax import random
from arviz_base.io_numpyro import from_numpyro_nested_mcmc

true_coefs = jnp.array([1., 2., 3.])
data = random.normal(random.PRNGKey(0), (2000, 3))
labels = dist.Bernoulli(logits=(true_coefs * data).sum(-1)).sample(random.PRNGKey(1))

def model(data, labels):
    with numpyro.plate("features", 3):
        coefs = numpyro.sample('coefs', dist.Normal(0, 1))
    intercept = numpyro.sample('intercept', dist.Normal(0., 10.))
    return numpyro.sample('y', dist.Bernoulli(logits=(coefs * data + intercept).sum(-1)),
                           obs=labels)

ns = NestedSampler(model)
ns.run(random.PRNGKey(2), data, labels,)
samples = ns.get_samples(random.PRNGKey(3), num_samples=1000)
assert jnp.mean(jnp.abs(samples['intercept'])) < 0.05
print(jnp.mean(samples['coefs'], axis=0))  # doctest: +SKIP


/Users/kylecaron/.pyenv/versions/az-dev/lib/python3.12/site-packages/jaxns/internals/mixed_precision.py:14: UserWarning: JAX x64 is not enabled. Setting it now. Check for errors.
  warnings.warn("JAX x64 is not enabled. Setting it now. Check for errors.")
INFO:2026-01-24 09:37:20,181:jax._src.xla_bridge:810: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: dlopen(libtpu.so, 0x0001): tried: 'libtpu.so' (no such file), '/System/Volumes/Preboot/Cryptexes/OSlibtpu.so' (no such file), '/Users/kylecaron/.pyenv/versions/3.12.11/lib/libtpu.so' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Users/kylecaron/.pyenv/versions/3.12.11/lib/libtpu.so' (no such file), '/opt/homebrew/lib/libtpu.so' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/lib/libtpu.so' (no such file), '/usr/lib/libtpu.so' (no such file, not in dyld cache), 'libtpu.so' (no such file)
INFO:jax._src.xla_bridge:Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: dl

[1.04410336 1.93030529 3.10239123]


In [23]:
samples['coefs'].shape

(1000, 3)

In [6]:
ns.constructor_kwargs

{'num_live_points': 100, 'devices': [CpuDevice(id=0)], 'max_samples': 10000.0}

In [7]:
from_numpyro_nested_mcmc(ns, model_args=(data, labels))

<xarray.DataTree>
Group: /
├── Group: /posterior
│       Dimensions:    (sample: 1000, features: 3)
│       Coordinates:
│         * sample     (sample) int64 8kB 0 1 2 3 4 5 6 ... 993 994 995 996 997 998 999
│         * features   (features) int64 24B 0 1 2
│       Data variables:
│           coefs      (sample, features) float64 24kB ...
│           intercept  (sample) float64 8kB ...
│       Attributes:
│           created_at:                 2026-01-24T14:42:27.695624+00:00
│           creation_library:           ArviZ
│           creation_library_version:   0.8.0dev0
│           creation_library_language:  Python
│           inference_library:          numpyro
│           inference_library_version:  0.19.0
└── Group: /observed_data
        Dimensions:  (y_dim_0: 2000)
        Coordinates:
          * y_dim_0  (y_dim_0) int64 16kB 0 1 2 3 4 5 ... 1994 1995 1996 1997 1998 1999
        Data variables:
            y        (y_dim_0) int64 16kB ...
        Attributes:
            created_at:                 2026-01-24T14:42:27.698612+00:00
            creation_library:           ArviZ
            creation_library_version:   0.8.0dev0
            creation_library_language:  Python
            inference_library:          numpyro
            inference_library_version:  0.19.0

In [24]:
from arviz_base.io_numpyro import BaseNumPyroConverter, SVIConverter

In [25]:
# ns.__dict__['_results']

In [26]:
# samples

In [27]:
ns

In [ ]:
class NestedMCMCConverter(SVIConverter):

    @property
    def model(self):
        if self.posterior is not None:
            return ns.model

    def _get_samples(self):
        """Extract samples from SVI guide."""
        return ns.get_samples(random.PRNGKey(0), num_samples=self.num_samples)



In [29]:
ns.get_samples(random.PRNGKey(0), num_samples=1000)['coefs'].shape

(1000, 3)

In [30]:
r = NestedMCMCConverter(ns, svi_result=None, model_args=(data, labels))
r.coords

In [34]:
from arviz_base.rcparams import rc_context, rcParams

with rc_context(rc={"data.sample_dims": ["sample"]}):
    idata = r.to_datatree()

In [ ]:
idata.

<xarray.DataTree>
Group: /
├── Group: /posterior
│       Dimensions:    (sample: 1000, features: 3)
│       Coordinates:
│         * sample     (sample) int64 8kB 0 1 2 3 4 5 6 ... 993 994 995 996 997 998 999
│         * features   (features) int64 24B 0 1 2
│       Data variables:
│           coefs      (sample, features) float64 24kB ...
│           intercept  (sample) float64 8kB ...
│       Attributes:
│           created_at:                 2026-01-24T13:55:56.830359+00:00
│           creation_library:           ArviZ
│           creation_library_version:   0.8.0dev0
│           creation_library_language:  Python
│           inference_library:          numpyro
│           inference_library_version:  0.19.0
└── Group: /observed_data
        Dimensions:  (y_dim_0: 2000)
        Coordinates:
          * y_dim_0  (y_dim_0) int64 16kB 0 1 2 3 4 5 ... 1994 1995 1996 1997 1998 1999
        Data variables:
            y        (y_dim_0) int64 16kB ...
        Attributes:
            created_at:                 2026-01-24T13:55:56.831806+00:00
            creation_library:           ArviZ
            creation_library_version:   0.8.0dev0
            creation_library_language:  Python
            inference_library:          numpyro
            inference_library_version:  0.19.0

In [4]:
r._get_samples()

NameError: name 'r' is not defined

In [26]:
from arviz_base.rcparams import rc_context, rcParams

with rc_context(rc={"data.sample_dims": ["sample"]}):

    idata = NestedMCMCConverter(ns, svi_result=None, model_args=(data, labels)).to_datatree()

(1000,)
0


In [29]:
idata.posterior['coefs']

<xarray.DataArray 'coefs' (sample: 1000, features: 3)> Size: 24kB
Array([[1.03786735, 1.83152328, 3.12585618],
       [1.0528507 , 1.93768731, 3.06501649],
       [1.06410361, 1.8232708 , 2.74788774],
       ...,
       [1.049478  , 2.07600546, 3.24774326],
       [1.11330267, 1.82043264, 3.13261383],
       [1.09512472, 1.74076621, 2.98168786]], dtype=float64)
Coordinates:
  * sample    (sample) int64 8kB 0 1 2 3 4 5 6 7 ... 993 994 995 996 997 998 999
  * features  (features) int64 24B 0 1 2